In this notebook,  there  are 2 set of of 100 randomly selected reviews:

1. The first 100 randomly selected reviews were selected from the entire data corpus  without any prior filtering
2. The second set of randomly 100 reviews, which is what is used for the ground truth  and testing the llms, was  selected from  the entire data corpus after prefiltering with 
a filtering list  in `filtering_keywords.py`.
3. 4  human annotators indepndently annotated the reviews following a criteria then a majority was created.
4.  8 of the llms  were  then used to a nnotaate the data and the top 3  were selected



In [ ]:
import sys
import os

# Add parent directory to Python path
sys.path.append(os.path.abspath(".."))

from scripts.utils import *

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa
import krippendorff
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')
import itertools
from collections import Counter


### Read in the data

In [ ]:
ANNOTATORS = ['ethel', 'theo', 'divya', 'brahms']
file_path = '/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/{ann}_100_swimreviews_labelled_12092025.csv'
merged_4ann = load_and_merge_data(ANNOTATORS, file_path)
print(display(merged_4ann.info()))

## Calculate fraction of reviews per label
1. For a  random selection of  100 reviews from the entire data corpus
2. For the random selection of 100 reviews from  a filtered subset of the enter data corpus (keyword filtering dictionary was used)

In [ ]:
# ----------------------------------------------------
# 1. FRACTIONS: Fraction of reviews that receive each label at least once
# ----------------------------------------------------
def compute_label_fractions(df):
    fractions = {}

    # Social
    social_yes = df[df['social'] == 'yes']['review_id'].nunique()
    fractions['social_yes'] = social_yes / df['review_id'].nunique()

    # Racial (among all reviews)
    racial_yes = df[df['racial'] == 'yes']['review_id'].nunique()
    fractions['racial_yes'] = racial_yes / df['review_id'].nunique()

    # Type-level (any label other than not_applicable)
    for t in ['discriminatory', 'pleasant', 'ambiguous']:
        type_count = df[df['type'] == t]['review_id'].nunique()
        fractions[f'type_{t}'] = type_count / df['review_id'].nunique()

    return fractions


#### The reviews were filtered with a keyword dictionary before a random selection is made for labeling

In [ ]:
# 1. Fractions
fractions = compute_label_fractions(merged_4ann)
print(fractions)

#### These reviews were selected randomly for labeling without an initial keyword filtering

In [ ]:
#no filtering
rev_nofiltered = pd.read_csv('/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/100_rsreviews_noflitering_allsc.csv')
# 1. Fractions
fractions = compute_label_fractions(rev_nofiltered)
print(fractions)

## 2. Calculating Agreement Scores across annotators

In [ ]:
# 3. Example Usage
# ----------------------------------------------------
# Define your categorical spaces
social_categories = ["Yes", "No"]
racial_categories = ["Yes", "No", "Not applicable"]
type_categories = ["Discriminatory", "Ambiguous", "Pleasant", "Not applicable"]

# For each variable, compute both scores:
for col, cats in [
    ("social", social_categories),
    ("racial", racial_categories),
    ("type", type_categories)
]:
    alpha_all, kappa_all = compute_agreement_scores(merged_4ann, col, cats, exclude_not_applicable=False)
    alpha_filtered, kappa_filtered = compute_agreement_scores(merged_4ann, col, cats, exclude_not_applicable=True)

    print(f"\n🟩 {col.upper()}")
    print(f"All (incl. 'Not applicable'): α={alpha_all:.4f}, κ={kappa_all:.4f}")
    print(f"Filtered (excl. 'Not applicable'): α={alpha_filtered:.4f}, κ={kappa_filtered:.4f}")


In [ ]:
merged_4ann.info()

## 3. Calculating the Majority

In [ ]:
# ----------------------------------------------------
# 3. MAJORITY VOTE GROUND TRUTH + tie export
# ----------------------------------------------------
def create_majority_vote(df, column, categories, adjudication_file):
    items, annotators, matrix = prepare_matrix(df, column, categories)

    majority_labels = {}
    ties = []

    for item, row in zip(items, matrix):
        vals = [v for v in row if pd.notna(v)]  # keep all valid values, don’t filter out categories
        if not vals:
            majority_labels[item] = 'not_applicable'
            continue

        counts = pd.Series(vals).value_counts()
        max_count = counts.max()

        if (counts == max_count).sum() > 1:
            # tie (2-way, 3-way, etc.)
            review_text = df.loc[df['review_id'] == item, 'trans_review'].iloc[0]
            ties.append({
                'review_id': item,
                'trans_review': review_text,
                'votes': counts.to_dict()
            })
            majority_labels[item] = np.nan
        else:
            majority_labels[item] = counts.idxmax()

    if ties:
        pd.DataFrame(ties).to_csv(adjudication_file, index=False)

    return majority_labels




In [ ]:
# 3. Majority vote (per level)
mv_social = create_majority_vote(merged_4ann, 'social', ['yes','no'], '/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/ties_social.csv')
mv_racial = create_majority_vote(merged_4ann, 'racial', ['yes','no','not_applicable'], '/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/ties_racial.csv')
mv_type   = create_majority_vote(merged_4ann, 'type', ['discriminatory','pleasant','ambiguous','not_applicable'], '/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/ties_type.csv')

In [ ]:
def find_majority_with_not_applicable(df, column, categories):
    items, annotators, matrix = prepare_matrix(df, column, categories)
    cases = []

    for item, row in zip(items, matrix):
        vals = [v for v in row if v in categories]
        if not vals:
            continue
        counts = pd.Series(vals).value_counts()
        # Check if not_applicable is present but not majority
        if "not_applicable" in counts.index and counts.idxmax() != "not_applicable":
            review_text = df.loc[df['review_id'] == item, 'trans_review'].iloc[0]
            cases.append({
                'review_id': item,
                'trans_review': review_text,
                'votes': counts.to_dict(),
                'majority': counts.idxmax()
            })

    return pd.DataFrame(cases)


In [ ]:
racial_cases = find_majority_with_not_applicable(
    merged_4ann, 'racial', ['yes','no','not_applicable']
)
racial_cases.to_csv("racial_majority_with_not_applicable.csv", index=False)


### 3.b Adjudicated labels

In [ ]:
ties_social_adj = pd.read_csv('/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/ties_social_adj.csv')
ties_racial_adj = pd.read_csv('/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/ties_racial_adj.csv')
ties_type_adj   = pd.read_csv('/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/ties_type_adj.csv')


In [ ]:
maj_social = pd.DataFrame(list(mv_social.items()), columns=['review_id', 'mv_social'])
maj_racial = pd.DataFrame(list(mv_racial.items()), columns=['review_id', 'mv_racial'])
maj_type   = pd.DataFrame(list(mv_type.items()), columns=['review_id', 'mv_type'])


In [ ]:
#replacing ties with adjudicated labels
for _, row in ties_social_adj.iterrows():
    maj_social.loc[maj_social['review_id'] == row['review_id'], 'mv_social'] = row['adj_social']

for _, row in ties_racial_adj.iterrows():
    maj_racial.loc[maj_racial['review_id'] == row['review_id'], 'mv_racial'] = row['adj_racial']

for _, row in ties_type_adj.iterrows():
    maj_type.loc[maj_type['review_id'] == row['review_id'], 'mv_type'] = row['adj_type']

## 4. Performance of other Human Annotators  & LLMs Against the ground truth

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
#merging all majority votes
ground_truth = maj_social.merge(maj_racial, on='review_id').merge(maj_type, on='review_id')
ground_truth.to_csv('/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/ground_truth_labels.csv', index=False)
ground_truth = ground_truth.rename(columns={'mv_social': 'social', 'mv_racial': 'racial', 'mv_type': 'type'})   
ground_truth

In [ ]:
ground_truth.nunique()

In [ ]:
humans = ['humanA', 'humanB']
llms = [
    'openai_gpt-4o-mini',
    'openai_gpt-4o', 
    'anthropic_claude-3.5-sonnet',
    'anthropic_claude-3-haiku',
    'google_gemini-2.5-pro',
    'google_gemini-2.5-flash',
    'deepseek_deepseek-chat',
    'deepseek_deepseek-r1-0528'
]

In [ ]:
all_annotators = humans + llms
base_path = '/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/'

ann_test_dfs = []

for ann in all_annotators:
    print(f"Processing {ann}...")
    paths_to_try = [
        f'{base_path}llms_results/100_swimreviews_{ann}_labels.csv',
        f'{base_path}{ann}_100_swimreviews_labelled_12092025.csv'
    ]
    df = None
    for path in paths_to_try:
        try:
            df = pd.read_csv(path)
            break
        except FileNotFoundError:
            continue
    if df is None:
        print(f"Error reading {ann}: file not found.")
        continue

    # Fill missing values
    df[['social', 'racial', 'type']] = df[['social', 'racial', 'type']].fillna('not_applicable')

    # Hierarchical logic
    df.loc[df['social'] == 'no', 'racial'] = 'not_applicable'
    df.loc[df['racial'].isin(['no', 'not_applicable']), 'type'] = 'not_applicable'

    df['annotator'] = ann
    # Option 1: Drop unnamed columns directly
    df = df.loc[:, df.columns.notnull()]  # keeps only named columns
    ann_test_dfs.append(df)

# Merge all dataframes once at the end
merged_df = pd.concat(ann_test_dfs, ignore_index=True)

# Drop unnecessary columns and duplicates
cols_to_drop = [col for col in ['Unnamed: 4', 'Unnamed: 5', ''] if col in merged_df.columns]
merged_df = merged_df.drop(columns=cols_to_drop)
merged_df = merged_df.loc[:, merged_df.columns.notnull()]  # keeps only named columns
merged_df = merged_df.drop_duplicates(subset=['review_id', 'annotator']).reset_index(drop=True)
merged_df =merged_df[['place_id', 'reviewer_id', 'trans_review','social','racial', 'type', 'annotator', 'model']]

print("Merged dataframe shape:", merged_df.shape)


In [ ]:
merged_df.info()

In [ ]:
# Updated function to compute metrics
def compute_metrics(gt, pred):
    metrics = {}
    # Map the column names we're using
    gt_columns = ['social_gt', 'racial_gt', 'type_gt']
    pred_columns = ['social_pred', 'racial_pred', 'type_pred']
    category_names = ['social', 'racial', 'type']
    
    for i, col in enumerate(category_names):
        metrics[col] = {
            'accuracy': accuracy_score(gt[gt_columns[i]], pred[pred_columns[i]]),
            'precision': precision_score(gt[gt_columns[i]], pred[pred_columns[i]], average='weighted', zero_division=0),
            'recall': recall_score(gt[gt_columns[i]], pred[pred_columns[i]], average='weighted', zero_division=0),
            'f1': f1_score(gt[gt_columns[i]], pred[pred_columns[i]], average='weighted', zero_division=0)
        }
    return metrics




In [ ]:
# Loop through annotators
results = {}
for df in ann_test_dfs:
    ann = df['annotator'].iloc[0]
    print(f"Evaluating {ann}...")

    
    # Merge with ground truth on review_id to align
    merged = ground_truth[['review_id','social','racial','type']].merge(
        df[['review_id','social','racial','type']], 
        on='review_id', 
        suffixes=('_gt','_pred')
    )
    
    print(f"Merged shape for {ann}: {merged.shape}")
    #print(merged.info())
    # Show only rows where at least one of these is NaN
    # Columns we care about
    check_cols = ['social_gt','social_pred','racial_gt','racial_pred','type_gt','type_pred']
 
    # Compute metrics - use the correct column names with suffixes
    metrics = compute_metrics(
        gt=merged[['social_gt','racial_gt','type_gt']],
        pred=merged[['social_pred','racial_pred','type_pred']]
    )
    results[ann] = metrics

# Convert results to DataFrame for easy viewing
metrics_df = pd.DataFrame({(i,j): results[i][j] 
                           for i in results.keys() 
                           for j in results[i].keys()}).T

print(metrics_df)

In [ ]:
# First, let's see what the actual structure of metrics_df looks like
print("Metrics DataFrame structure:")
print(metrics_df.head())
print("\nColumns:", metrics_df.columns.tolist())
print("\nIndex levels:", metrics_df.index.names if hasattr(metrics_df.index, 'names') else "No multiindex")

# If it's a MultiIndex, let's see what's in it
if hasattr(metrics_df.index, 'levels'):
    print("Index levels content:")
    for i, level in enumerate(metrics_df.index.levels):
        print(f"Level {i}: {level.tolist()}")

### 4.b Visualise the performance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set up the plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("Set2")

In [ ]:

# First, let's properly prepare the metrics_df
print("Original metrics_df structure:")
print(metrics_df.head())

# Reset index to get annotator and category as columns
metrics_df_reset = metrics_df.reset_index()

# Rename columns for clarity
metrics_df_reset = metrics_df_reset.rename(columns={'level_0': 'annotator', 'level_1': 'category'})

print("Columns after reset_index:", metrics_df_reset.columns.tolist())

print("\nPrepared DataFrame:")
print(metrics_df_reset.head())
print("\nDataFrame columns:", metrics_df_reset.columns.tolist())

In [ ]:
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

def compute_auc(gt, pred_probs, classes):
    # Binarize GT labels for multi-class ROC
    y_true = label_binarize(gt, classes=classes)
    n_classes = y_true.shape[1]

    auc_scores = {}
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true[:, i], pred_probs[:, i])
        auc_scores[classes[i]] = auc(fpr, tpr)
    
    # Macro average
    auc_scores['macro'] = roc_auc_score(y_true, pred_probs, multi_class='ovr', average='macro')
    return auc_scores


In [ ]:
import seaborn as sns
plt.figure(figsize=(10,6))
sns.heatmap(metrics_df.unstack(level=1), annot=True, cmap="Blues")
plt.title("Performance of Annotators across Categories")
plt.show()

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(metrics_df, annot=True, cmap="Blues", fmt=".2f")
plt.title("Performance Heatmap: Annotators × Categories × Metrics")
plt.show()